By: Jesse Anderson

Implementation of water depth / bathymetry algorithm as described here: https://www.mdpi.com/2072-4292/13/8/14691024

Code translated from GEE code here: https://github.com/CoralMapping/GEE_Sentinel2_Bathymetry_Paper

In [ ]:
import rioxarray as rx
import xarray as xr
from pystac_client import Client
from odc.stac import load
from math import exp
import numpy as np

In [ ]:
def calculate_depth(xarr):
    bigrrs = xarr / 31415.926
    rrsvec = bigrrs / (bigrrs * 1.7 + 0.52)
    rrsvec1k = rrsvec * 1000

    # This is the chlorophyll-a level, and may be too high for a lot of locations.
    # It's worth trying different values to see how results change
    chla = 0.5
    m0 = 52.073 * exp(0.957 * chla)
    m1 = 50.156 * exp(0.957 * chla)

    lnrrsvec = np.log(rrsvec1k)
    blue = lnrrsvec.B02
    green = lnrrsvec.B03
    depth = (blue / green) * m0 - m1

    # This clamping is in the original paper. It's worth going without, or leaving
    # extreme values as nan to see where estimates are saturated at high or low values.
    return depth.where(depth > 0, 0).where(depth < 20, 20).where(~xarr.B02.isnull())

In [ ]:
# Bounds for Samoa
bbox = [-172.8, -13.8, -172.1, -13.37]

catalog = Client.open(
    "https://earth-search.aws.element84.com/v1",
)

search = catalog.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=bbox,
    datetime="2024-01/2024-06",
    query={"eo:cloud_cover": {"lt": 10}},
)

In [ ]:
data = load(
    search.items(),
    chunks=dict(x=2048, y=2048),
    bbox=bbox,
    crs=3832,
    bands=["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09", "scl"],
    group_by="solar_day",
)

# nodata, cloud shadow, medium cloud, high cloud
mask_flags = [1, 3, 8, 9]
cloud_mask = ~data.scl.isin(mask_flags)
masked = data.where(cloud_mask).drop_vars("scl")

scaled = (masked.where(masked != 0) * 0.0001).clip(0, 1)

scaled = masked

In [ ]:
# Create a median
median = masked.median(dim="time").compute()
median

In [ ]:
sdb = calculate_depth(median_masked).to_dataset(name="depth").compute()

sdb

In [ ]:
sdb.depth.odc.explore(cmap="Blues", robust=True)